In [5]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bethesda-classification")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: http://127.0.0.1:5000


## Служебная информация об эксперименте

Фиксируем имя участника, название задачи, дату и текущий Git commit.

Git commit используется как идентификатор версии кода. Перед запуском модели
необходимо убедиться, что код эксперимента закоммичен. Если в рабочей директории
есть незакоммиченные изменения, текущий commit не полностью описывает состояние
кода, на котором выполняется эксперимент.

In [6]:
import subprocess
from datetime import datetime


participant = "Savely"
task_name = "bethesda-classification"

# Получаем SHA текущего Git-коммита
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

# Проверяем наличие незакоммиченных изменений
git_status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    text=True,
).strip()

run_date = datetime.now().strftime("%d-%m-%Y")

print("Participant:", participant)
print("Task:", task_name)
print("Date:", run_date)
print("Git commit:", git_commit)

if git_status:
    print("\nWARNING: в репозитории есть незакоммиченные изменения.")
    print("Перед первым MLflow run необходимо сделать commit и повторно выполнить эту ячейку.")
else:
    print("\nGit working tree clean.")

Participant: Savely
Task: bethesda-classification
Date: 14-10-2026
Git commit: bd1689c42c15ce2e7f451f9d394462961b555094

Перед первым MLflow run необходимо сделать commit и повторно выполнить эту ячейку.


## Версия датасета

Для экспериментов используется актуальный `prepared_dataset.csv`.

Версия данных определяется полным SHA256-хешем файла. Если содержимое датасета
изменится хотя бы на уровне одного байта, изменится и его SHA256.

Полный SHA256 будет сохраняться в каждом MLflow run как `dataset_version`.
Первые 12 символов используются только для удобного отображения и имени
архивной копии датасета.

In [7]:
from pathlib import Path

from src.common.dataset_versioning import calculate_sha256


# Определяем корень проекта
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = next(
        path
        for path in PROJECT_ROOT.parents
        if (path / "src").exists()
    )

# Актуальный воспроизводимый датасет
dataset_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "prepared_dataset.csv"
)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Датасет не найден: {dataset_path}"
    )

# Вычисляем версию данных
dataset_hash = calculate_sha256(dataset_path)
dataset_version_short = dataset_hash[:12]

# Ожидаемая архивная копия этой версии
dataset_version_path = (
    dataset_path.parent
    / "versions"
    / f"{dataset_path.stem}_{dataset_version_short}{dataset_path.suffix}"
)

print("Dataset:", dataset_path.name)
print("Dataset version:", dataset_version_short)
print("Full SHA256:", dataset_hash)
print("Archived version:", dataset_version_path.name)
print("Archive exists:", dataset_version_path.exists())

Dataset: prepared_dataset.csv
Dataset version: e88b34b729ea
Full SHA256: e88b34b729eaf057b6f2a7deeae7fcfce901da710ddce628a329d9d96aaad724
Archived version: prepared_dataset_e88b34b729ea.csv
Archive exists: True


## Загрузка и первичная проверка датасета

Загрузим актуальный `prepared_dataset.csv`, для которого выше был вычислен
SHA256-хеш.

Проверим размер датасета, структуру столбцов и первые строки. На этом этапе
модель ещё не обучается, поэтому отдельный MLflow run не создаётся.

In [8]:
import pandas as pd

df = pd.read_csv(dataset_path, sep=";")

print("Путь к датасету:", dataset_path)
print("Размер датасета:", df.shape)
print("Столбцы:", df.columns.tolist())

Путь к датасету: /Users/savely/Desktop/diploma/data/processed/prepared_dataset.csv
Размер датасета: (5613, 8)
Столбцы: ['case_id', 'source_table', 'source_row', 'description', 'conclusion', 'bethesda', 'duplicate_group', 'duplicate_type']


## Проверка структуры и качества данных

Перед построением модели проверим:

- наличие необходимых столбцов;
- количество пропущенных значений;
- число объектов каждого класса Bethesda;
- долю каждого класса в датасете.

Это позволит оценить состояние нового датасета и степень дисбаланса классов
перед разделением на train и test.

In [9]:
print("Столбцы:")
print(df.columns.tolist())

print("\nПропущенные значения:")
print(df.isna().sum())

print("\nКоличество объектов каждого класса Bethesda:")
print(df["bethesda"].value_counts().sort_index())

print("\nДоля каждого класса:")
print(
    df["bethesda"]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Столбцы:
['case_id', 'source_table', 'source_row', 'description', 'conclusion', 'bethesda', 'duplicate_group', 'duplicate_type']

Пропущенные значения:
case_id               0
source_table          0
source_row            0
description           0
conclusion            0
bethesda              0
duplicate_group    5603
duplicate_type     5603
dtype: int64

Количество объектов каждого класса Bethesda:
bethesda
I       575
II     3825
III     286
IV      559
V       172
VI      196
Name: count, dtype: int64

Доля каждого класса:
bethesda
I      0.1024
II     0.6815
III    0.0510
IV     0.0996
V      0.0306
VI     0.0349
Name: proportion, dtype: float64


## Проверка дубликатов

В актуальном датасете информация о найденных дубликатах сохраняется в полях
`duplicate_group` и `duplicate_type`.

Дубликаты не удаляются автоматически на этапе подготовки данных. Перед обучением
необходимо проверить найденные группы и исключить ситуацию, когда одинаковые
описания попадают одновременно в обучающую и тестовую выборки.

Дополнительно проверим уникальность `case_id`, наличие полных дубликатов строк
и повторяющихся значений `description`.

In [10]:
# Проверяем уникальность идентификаторов
duplicate_case_ids = df["case_id"].duplicated().sum()

# Проверяем полные дубликаты строк
duplicate_rows = df.duplicated().sum()

# Проверяем одинаковые description
duplicate_descriptions = df["description"].duplicated().sum()

# Выбираем записи, для которых задана группа дубликатов
marked_duplicates = df[
    df["duplicate_group"].notna()
].copy()

print("Дубликаты case_id:", duplicate_case_ids)
print("Полные дубликаты строк:", duplicate_rows)
print("Повторяющиеся description:", duplicate_descriptions)

print("\nСтрок с duplicate_group:", len(marked_duplicates))
print(
    "Количество групп дубликатов:",
    marked_duplicates["duplicate_group"].nunique()
)

print("\nТипы дубликатов:")
print(
    marked_duplicates["duplicate_type"]
    .value_counts(dropna=False)
)

Дубликаты case_id: 0
Полные дубликаты строк: 0
Повторяющиеся description: 0

Строк с duplicate_group: 10
Количество групп дубликатов: 10

Типы дубликатов:
duplicate_type
technical    10
Name: count, dtype: int64


## Разделение данных на обучающую и тестовую выборки

Для классификации используется только поле `description`.
Поле `bethesda` является целевой переменной.

Служебные поля `case_id`, `source_table`, `source_row`, `duplicate_group`
и `duplicate_type`, а также поле `conclusion`, в признаки модели не включаются.

Перед разделением было подтверждено отсутствие повторяющихся `description`,
поэтому одинаковые тексты не могут одновременно попасть в train и test.

Данные разделяются в отношении 80/20 со стратификацией по Bethesda.
Параметр `random_state=42` фиксируется для воспроизводимости эксперимента.

In [11]:
from sklearn.model_selection import train_test_split


# Вход модели - только цитологическое описание
X = df["description"]

# Целевая переменная - категория Bethesda
y = df["bethesda"]

# Стратифицированное разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Размер всей выборки:", len(X))
print("Размер train:", len(X_train))
print("Размер test:", len(X_test))

print("\nРаспределение классов в train:")
print(y_train.value_counts().sort_index())

print("\nРаспределение классов в test:")
print(y_test.value_counts().sort_index())

Размер всей выборки: 5613
Размер train: 4490
Размер test: 1123

Распределение классов в train:
bethesda
I       460
II     3060
III     229
IV      447
V       137
VI      157
Name: count, dtype: int64

Распределение классов в test:
bethesda
I      115
II     765
III     57
IV     112
V       35
VI      39
Name: count, dtype: int64



## Загрузка обученных моделей из MLflow

Для сравнительного анализа используются две ранее обученные модели:

1. **Logistic Regression** - эксперимент №7, `C=4.0`.
2. **Linear SVM** - эксперимент №2, `C=0.5`.

Обе модели используют объединённые Word TF-IDF и Character TF-IDF признаки.

Модели загружаются по идентификаторам экспериментов MLflow.
Повторное обучение не выполняется.

Перед загрузкой проверяется соответствие SHA256 датасета
версии данных, зафиксированной в каждом эксперименте.

Цель этапа - восстановить сохранённые классификаторы
для последующего сравнения их предсказаний на одной
тестовой выборке.


In [12]:
import mlflow
import mlflow.sklearn


# Run ID лучших экспериментов
LOGREG_RUN_ID = "7d0f146786c248e2989a58898dc4b7f9"
SVM_RUN_ID = "7d7cded6b14d4867b57a75f9d7550d24"

# Получаем эксперимент из MLflow
experiment = mlflow.get_experiment_by_name(
    "bethesda-classification"
)

if experiment is None:
    raise RuntimeError("Эксперимент MLflow не найден.")


def load_model_from_run(run_id):
    # Проверяем версию датасета
    run_info = mlflow.get_run(run_id)
    saved_hash = run_info.data.params.get("dataset_version")

    if saved_hash != dataset_hash:
        raise ValueError(
            f"SHA256 датасета не совпадает для run {run_id}"
        )

    # Находим сохранённую модель по Run ID
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"source_run_id = '{run_id}'",
        output_format="list",
    )

    # Выбираем модель, сохранённую под именем model
    models = [m for m in models if m.name == "model"]

    if len(models) != 1:
        raise RuntimeError(
            f"Для run {run_id} найдено моделей: {len(models)}"
        )

    # Загружаем готовый Pipeline
    model_uri = f"models:/{models[0].model_id}"

    return mlflow.sklearn.load_model(model_uri)


# Загружаем обе модели
model_logreg = load_model_from_run(LOGREG_RUN_ID)
model_svm = load_model_from_run(SVM_RUN_ID)

print("Обе модели успешно загружены!")
print("Logistic Regression:", model_logreg.steps[-1][1])
print("Linear SVM:", model_svm.steps[-1][1])

Обе модели успешно загружены!
Logistic Regression: LogisticRegression(C=4.0, class_weight='balanced', max_iter=1000,
                   random_state=42)
Linear SVM: LinearSVC(C=0.5, class_weight='balanced', max_iter=10000, random_state=42)


## Сравнение ошибочных предсказаний Logistic Regression и Linear SVM

На данном этапе обе обученные модели применяются к одной
и той же тестовой выборке.

Цель анализа - определить:

- Количество ошибок каждой модели.
- Количество записей, на которых ошиблись обе модели.
- Количество ошибок, характерных только для Logistic Regression.
- Количество ошибок, характерных только для Linear SVM.
- Общее количество уникальных записей с ошибками.

Результаты позволят оценить, насколько различается
поведение классификаторов, несмотря на близкие значения
метрик качества.

Для дальнейшего ручного анализа формируется единая
таблица предсказаний без изменения исходного датасета.

In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, balanced_accuracy_score


# Получаем предсказания сохранённых моделей
y_pred_logreg = model_logreg.predict(X_test)
y_pred_svm = model_svm.predict(X_test)

# Приводим правильные ответы к массиву
y_true = np.asarray(y_test)

# Формируем общую таблицу предсказаний
comparison_df = pd.DataFrame({
    "test_position": np.arange(len(y_true)),
    "bethesda_true": y_true,
    "logreg_pred": y_pred_logreg,
    "svm_pred": y_pred_svm,
})

# Отмечаем ошибки каждой модели
comparison_df["logreg_error"] = (
    comparison_df["bethesda_true"]
    != comparison_df["logreg_pred"]
)

comparison_df["svm_error"] = (
    comparison_df["bethesda_true"]
    != comparison_df["svm_pred"]
)

# Разделяем записи по типу ошибки
comparison_df["error_group"] = np.select(
    [
        comparison_df["logreg_error"]
        & comparison_df["svm_error"],

        comparison_df["logreg_error"]
        & ~comparison_df["svm_error"],

        ~comparison_df["logreg_error"]
        & comparison_df["svm_error"],
    ],
    [
        "Обе модели ошиблись",
        "Только Logistic Regression",
        "Только Linear SVM",
    ],
    default="Обе модели верны",
)

# Подсчитываем ошибки
errors_logreg = int(comparison_df["logreg_error"].sum())
errors_svm = int(comparison_df["svm_error"].sum())

common_errors = int((
    comparison_df["logreg_error"]
    & comparison_df["svm_error"]
).sum())

unique_errors = int((
    comparison_df["logreg_error"]
    | comparison_df["svm_error"]
).sum())

# Проверяем воспроизведение сохранённых метрик
print("Logistic Regression:")
print("Macro-F1:", round(f1_score(y_true, y_pred_logreg,
                                 average="macro"), 4))
print("Ошибок:", errors_logreg)

print("\nLinear SVM:")
print("Macro-F1:", round(f1_score(y_true, y_pred_svm,
                                 average="macro"), 4))
print("Ошибок:", errors_svm)

print("\nРаспределение ошибок:")
print(comparison_df["error_group"].value_counts())

print("\nОбщих ошибок:", common_errors)
print("Уникальных ошибочных записей:", unique_errors)

# Проверяем согласованность подсчётов
assert errors_logreg == 12
assert errors_svm == 12

print("\nПроверка количества ошибок пройдена.")

Logistic Regression:
Macro-F1: 0.9725
Ошибок: 12

Linear SVM:
Macro-F1: 0.9717
Ошибок: 12

Распределение ошибок:
error_group
Обе модели верны              1109
Обе модели ошиблись             10
Только Logistic Regression       2
Только Linear SVM                2
Name: count, dtype: int64

Общих ошибок: 10
Уникальных ошибочных записей: 14

Проверка количества ошибок пройдена.


In [14]:
import numpy as np
import pandas as pd


# Отбираем записи, где ошиблась хотя бы одна модель
errors_df = comparison_df.loc[
    comparison_df["logreg_error"]
    | comparison_df["svm_error"]
].copy()

# Восстанавливаем индекс строки исходного датасета
errors_df["dataset_index"] = np.asarray(
    X_test.index
)[errors_df["test_position"].to_numpy()]

# Добавляем медицинские описания для ручного анализа
errors_df["description"] = np.asarray(
    X_test
)[errors_df["test_position"].to_numpy()]

# Проверяем различие предсказаний моделей
errors_df["predictions_differ"] = (
    errors_df["logreg_pred"] != errors_df["svm_pred"]
)

# Проверяем, одинаково ли ошиблись обе модели
errors_df["same_wrong_prediction"] = (
    errors_df["logreg_error"]
    & errors_df["svm_error"]
    & ~errors_df["predictions_differ"]
)

# Выводим только служебные поля без медицинских текстов
columns_to_show = [
    "test_position",
    "dataset_index",
    "bethesda_true",
    "logreg_pred",
    "svm_pred",
    "error_group",
    "predictions_differ",
]

print("Количество ошибочных записей:", len(errors_df))

print(
    "Одинаковые неправильные предсказания:",
    int(errors_df["same_wrong_prediction"].sum())
)

print(
    "Разные предсказания:",
    int(errors_df["predictions_differ"].sum())
)

display(
    errors_df[columns_to_show].reset_index(drop=True)
)

Количество ошибочных записей: 14
Одинаковые неправильные предсказания: 10
Разные предсказания: 4


,test_position,dataset_index,bethesda_true,logreg_pred,svm_pred,error_group,predictions_differ
0,97,269,II,I,II,Только Logistic Regression,True
1,125,5388,V,VI,VI,Обе модели ошиблись,False
2,373,1673,II,IV,II,Только Logistic Regression,True
3,581,1794,I,I,II,Только Linear SVM,True
4,647,1260,IV,V,V,Обе модели ошиблись,False
5,670,4475,I,III,III,Обе модели ошиблись,False
6,696,2333,III,I,I,Обе модели ошиблись,False
7,726,3788,V,IV,IV,Обе модели ошиблись,False
8,768,3397,IV,V,V,Обе модели ошиблись,False
9,854,1236,I,IV,IV,Обе модели ошиблись,False


## Ручной анализ ошибочных классификаций

На тестовой выборке выявлено 14 уникальных записей,
на которых хотя бы одна из моделей допустила ошибку.

Из них:
- 10 записей ошибочно классифицированы обеими моделями;
- 2 записи ошибочно классифицированы только Logistic Regression;
- 2 записи ошибочно классифицированы только Linear SVM.

Всего зафиксировано 24 неправильных предсказания.

Цель анализа - выявить возможные причины ошибок:

- Наличие неоднозначных медицинских формулировок.
- Недостаток характерных текстовых признаков.
- Сходство морфологических признаков разных категорий.
- Особенности обработки отрицаний и сокращений.
- Возможные проблемы исходной разметки.
- Ограничения TF-IDF при интерпретации контекста.

In [25]:
from pathlib import Path

# Папка для локального хранения результатов
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Путь к файлу с ошибочными классификациями
output_path = output_dir / "classification_error_cases.txt"

# Сортируем записи по индексу датасета
problem_cases = errors_df.sort_values(
    "dataset_index"
).reset_index(drop=True)

# Формируем TXT с полными сведениями об ошибках
with open(output_path, "w", encoding="utf-8") as file:

    file.write("АНАЛИЗ ОШИБОК КЛАССИФИКАЦИИ BETHESDA\n")
    file.write("Модели: Logistic Regression и Linear SVM\n")
    file.write(f"Количество записей: {len(problem_cases)}\n\n")

    for number, row in problem_cases.iterrows():

        file.write(f"\nСЛУЧАЙ {number + 1}\n")

        file.write(
            f"Индекс в датасете: {row['dataset_index']}\n"
        )
        file.write(
            f"Истинная Bethesda: {row['bethesda_true']}\n"
        )
        file.write(
            f"Logistic Regression: {row['logreg_pred']}\n"
        )
        file.write(
            f"Linear SVM: {row['svm_pred']}\n"
        )
        file.write(
            f"Тип ошибки: {row['error_group']}\n"
        )

        file.write("\nЦитологическое описание:\n")
        file.write(str(row["description"]) + "\n\n")

# Выводим только подтверждение сохранения
print("Файл успешно сохранён.")
print("Путь:", output_path)
print("Количество записей:", len(problem_cases))

Файл успешно сохранён.
Путь: /Users/savely/Desktop/diploma/data/processed/classification_error_cases.txt
Количество записей: 14
